## Silhouette Score senza etichette

Silhouette misura quanto, dato un clustering non supervisionato, i punti stanno vicini ai propri vicini e lontani dagli altri. Più è alto (da –1 a +1), meglio l’embedding separa naturalmente i mazzi.

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from structures.deck import Deck
Embedding = Deck.Embedding
import pickle
import random
from pathlib import Path

Creiamo un Sample di mazzi su cui testare

In [ ]:
PICKLE_PATH = Path("cache/decks.pkl")

with open(PICKLE_PATH, "rb") as f:
    training_sample = random.sample(pickle.load(f), 2000)

In [ ]:
def eval_silhouette(embedding: Embedding, decks: list[Deck],
                    n_clusters: int = 50, metric: str = "cosine") -> float:
    # Estrai matrice N×D
    X = np.vstack([np.array(embedding(d)) for d in decks])
    # ( opzionale: riduci con PCA se D è troppo grande )
    labels = KMeans(n_clusters, random_state=0).fit_predict(X)
    return silhouette_score(X, labels, metric=metric)

# Lista di tutti gli embedding disponibili
all_embs = [
    Embedding.QUANTITY,
    Embedding.KEYWORDS,
    Embedding.SUPERTYPES,
    Embedding.SUBTYPES,
    Embedding.PKMN_TYPES,
    Embedding.EVO_STATS,
    Embedding.WEAKNESS,
    Embedding.RESISTANCE,
    Embedding.HP,
    Embedding.ATTACKS_DMG,
    Embedding.ATTACKS_COSTS
]

# Esempio di valutazione
scores = {
    e: eval_silhouette(e, sample, n_clusters=30)
    for e in all_embs
}
# poi ordini `scores` per valore decrescente per individuare l'embedding con valore più alto


QUANTITY (vettore giganorme di conteggi) dà massima granularità, \\
PKMN_TYPES–EVO_STATS–WEAKNESS–RESISTANCE sono vettori cortissimi e danno insight sulla “struttura” del mazzo,\\
KEYWORDS cattura il testo delle abilità/regole

Un embedding “buono” deve raggruppare i mazzi veramente simili molto vicini fra loro, e non comprimere tutto in uno spazio uniforme.

In [ ]:
import numpy as np
from sklearn.neighbors import NearestNeighbors

def eval_nn_density(embedding: D.Embedding, decks: list[Deck], k: int = 5):
    X = np.vstack([np.array(embedding(d)) for d in decks])
    # Se usi Similarity.COSENO, normalizza
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    Xn = X / np.where(norms==0, 1, norms)
    nbrs = NearestNeighbors(n_neighbors=k+1, metric="cosine").fit(Xn)
    dists, _ = nbrs.kneighbors(Xn)
    # dists[:, 0] = 0 (sé stessi), prendi media dei k vicini veri
    return np.mean(dists[:, 1:])

densities = {
    e.name: eval_nn_density(e, sample_decks, k=5)
    for e in all_embs
}
# valori più bassi indicano vicini più “stretti”


Per capire se un embedding è “ricco” o “rumoroso”, puoi ridurlo con una PCA e vedere quanti componenti servono per raggiungere ad esempio il 90 % di varianza:

In [ ]:
from sklearn.decomposition import PCA

def pca_var_explained(embedding: D.Embedding, decks: list[Deck], thresh: float = 0.9):
    X = np.vstack([np.array(embedding(d)) for d in decks])
    pca = PCA().fit(X)
    cumvar = np.cumsum(pca.explained_variance_ratio_)
    # primo indice i tale che cumvar[i] ≥ thresh
    return np.searchsorted(cumvar, thresh) + 1

pca_dims = {
    e.name: pca_var_explained(e, sample_decks)
    for e in all_embs
}
